In [ ]:
%cd ../../

In [ ]:
import sys
import math
from pathlib import Path

import yaml
import polars as pl
import numpy as np
from loguru import logger
# from transformers import BertModel  # BertTokenizer
from transformers import AutoTokenizer
from polars import DataFrame
from tqdm import tqdm

In [ ]:
logger.remove()
logger.add(sys.stderr, level="DEBUG")

# Load things

In [ ]:
path = "src/gen_retrieval/configs.yaml"

with open(path) as file:
    conf = yaml.safe_load(file)

conf

In [ ]:
MODEL_NAME = conf['MODEL_DOCID']

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = BertModel.from_pretrained(MODEL_NAME).to(device="mps")

In [ ]:
corpus = pl.read_ndjson(conf['RAW_DATA']['corpus'])
corpus.head()

# Tokenize entire corpus

In [ ]:
def _get_text(corpus: DataFrame):
    for row in corpus.iter_rows(named=True):
        yield row['text']

iter = _get_text(corpus)
bsz = 200
total = math.ceil(len(corpus) / bsz)

input_ids, token_type_ids, attention_mask = [], [], []
idx = 0
with tqdm(total=total) as pbar:
    while True:
        n = bsz if idx < total - 1 else len(corpus) - bsz * (total - 1)
        doc = [next(iter) for _ in range(n)]

        encoded_input = tokenizer(doc, return_tensors='np', padding='max_length', truncation=True, max_length=512)
        input_ids.append(encoded_input['input_ids'])
        # token_type_ids.append(encoded_input['token_type_ids'])
        attention_mask.append(encoded_input['attention_mask'])
        # output = model(**encoded_input).last_hidden_state.mean(dim=1)
        # embds.append(output)

        pbar.update(1)
        idx += 1

        if idx == total:
            break

In [ ]:
path = Path(conf['INTERIM']['docid'])
path.parent.mkdir(exist_ok=True, parents=True)

np.savez(
    path,
    {
        'input_ids': np.vstack(input_ids),
        'attention_mask': np.vstack(attention_mask),
    }
)